In [1]:
import qiskit_ionq
from helpers import get_ionq_api_key

In [2]:
import numpy as np
from scipy.optimize import minimize

from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp, Operator

In [3]:
api_key = get_ionq_api_key()
provider = qiskit_ionq.IonQProvider(token=api_key)

/home/gserpa/01 Projects/ion_q_sandbox/.venv/lib/python3.11/site-packages/qiskit_ionq/ionq_backend.py:127: IonQTranspileLevelWarning: Transpiler default optimization_level=2. IonQ (QIS) recommends 0-1 to avoid aggressive re-synthesis; use transpile(..., optimization_level=1).
  warn_bad_transpile_level()


In [4]:
for item in provider.backends():
    print(item)

In [5]:
def create_universal_ansatz(num_qubits, num_layers):
    theta = ParameterVector("θ", num_qubits * 3 * num_layers + num_qubits * 2 * (num_layers - 1))
    ansatz = QuantumCircuit(num_qubits)
    
    param_idx = 0
    for layer in range(num_layers):
        # Single-qubit rotations: Ry, Rz on each qubit
        for q in range(num_qubits):
            ansatz.ry(theta[param_idx], q)
            param_idx += 1
            ansatz.rz(theta[param_idx], q)
            param_idx += 1
        
        # Entangling layer (CX ladder)
        if layer < num_layers - 1:
            for q in range(num_qubits - 1):
                ansatz.cx(q, q + 1)
            # Optional: wrap around
            if num_qubits > 2:
                ansatz.cx(num_qubits - 1, 0)
    
    # Final single-qubit rotations
    for q in range(num_qubits):
        ansatz.ry(theta[param_idx], q)
        param_idx += 1
        ansatz.rz(theta[param_idx], q)
        param_idx += 1
    
    return ansatz, theta

In [6]:
# What follows is a usage example
ansatz, theta = create_universal_ansatz(2, num_layers=2)
print(ansatz.draw())

     ┌──────────┐┌──────────┐     ┌──────────┐┌──────────┐ ┌──────────┐»
q_0: ┤ Ry(θ[0]) ├┤ Rz(θ[1]) ├──■──┤ Ry(θ[4]) ├┤ Rz(θ[5]) ├─┤ Ry(θ[8]) ├»
     ├──────────┤├──────────┤┌─┴─┐├──────────┤├──────────┤┌┴──────────┤»
q_1: ┤ Ry(θ[2]) ├┤ Rz(θ[3]) ├┤ X ├┤ Ry(θ[6]) ├┤ Rz(θ[7]) ├┤ Ry(θ[10]) ├»
     └──────────┘└──────────┘└───┘└──────────┘└──────────┘└───────────┘»
«      ┌──────────┐
«q_0: ─┤ Rz(θ[9]) ├
«     ┌┴──────────┤
«q_1: ┤ Rz(θ[11]) ├
«     └───────────┘


In [7]:
def hardware_efficient_ansatz(num_qubits, num_layers):
    theta = ParameterVector("θ", num_qubits * 2 * num_layers)
    ansatz = QuantumCircuit(num_qubits)
    
    param_idx = 0
    for layer in range(num_layers):
        for q in range(num_qubits):
            ansatz.ry(theta[param_idx], q)
            param_idx += 1
            ansatz.rz(theta[param_idx], q)
            param_idx += 1
        for q in range(num_qubits - 1):
            ansatz.cx(q, q + 1)
    
    return ansatz, theta

In [8]:
# Usage example
ansatz, theta = hardware_efficient_ansatz(2, num_layers=2)
print(ansatz.draw())

     ┌──────────┐┌──────────┐     ┌──────────┐┌──────────┐     
q_0: ┤ Ry(θ[0]) ├┤ Rz(θ[1]) ├──■──┤ Ry(θ[4]) ├┤ Rz(θ[5]) ├──■──
     ├──────────┤├──────────┤┌─┴─┐├──────────┤├──────────┤┌─┴─┐
q_1: ┤ Ry(θ[2]) ├┤ Rz(θ[3]) ├┤ X ├┤ Ry(θ[6]) ├┤ Rz(θ[7]) ├┤ X ├
     └──────────┘└──────────┘└───┘└──────────┘└──────────┘└───┘


## Define the Hamiltonian

In [9]:
# Simple H = X x Y
H = SparsePauliOp("XY")   

# First just the simple example, no obfuscation 

## Print the exact spectrum

In [10]:
# Print exact spectrum
eigvals = np.linalg.eigvalsh(Operator(H).data)
print(f"Exact eigenvalues: {np.round(eigvals, 6)}")
print(f"Exact ground state energy: {eigvals[0]:.6f}\n")

Exact eigenvalues: [-1. -1.  1.  1.]
Exact ground state energy: -1.000000



# Hardware efficient ansatz for 2 qubits

In [11]:
ansatz, theta = hardware_efficient_ansatz(2, num_layers=2)
print(ansatz.draw())

     ┌──────────┐┌──────────┐     ┌──────────┐┌──────────┐     
q_0: ┤ Ry(θ[0]) ├┤ Rz(θ[1]) ├──■──┤ Ry(θ[4]) ├┤ Rz(θ[5]) ├──■──
     ├──────────┤├──────────┤┌─┴─┐├──────────┤├──────────┤┌─┴─┐
q_1: ┤ Ry(θ[2]) ├┤ Rz(θ[3]) ├┤ X ├┤ Ry(θ[6]) ├┤ Rz(θ[7]) ├┤ X ├
     └──────────┘└──────────┘└───┘└──────────┘└──────────┘└───┘


## Define the backend

In [12]:
# --- IonQ simulator (free, no QPU cost) ---
# backend = provider.get_backend('ionq_simulator')
# backend.set_options(noise_model="forte-1")

# --- To run on Forte 1 QPU, comment the line above and uncomment below ---
backend = provider.get_backend('qpu.forte-1')

## QPU cost control

Three parameters control how many circuit jobs are submitted per optimization run:

- **`nshots`** — shots per circuit execution. More shots = less statistical noise but higher cost per job.
- **`maxiter`** — max optimizer iterations per trial. Each iteration submits one circuit per Pauli term in H.
- **`range(N)`** in the optimization loop — number of random-restart trials. Each trial runs a full optimization.

**Worst-case job count:** `trials × maxiter × num_pauli_terms`

Current settings (simulator): `3 × 300 × 1 = 900` jobs.

Suggested QPU starting point: `range(1)`, `maxiter=50`, `nshots=1024` → 50 jobs for H=XY.

# Cost function

In [13]:
nshots = 500

def compute_pauli_expectation(bound_circuit, pauli_label, backend, shots):
    """Measure <pauli_label> for a bound circuit on a real/simulated backend."""
    qc = bound_circuit.copy()
    n = qc.num_qubits

    # Basis-change gates so Z-measurement reads the Pauli eigenbasis
    for i, p in enumerate(pauli_label):
        qubit = n - 1 - i  # pauli_label[0] -> highest qubit
        if p == 'X':
            qc.h(qubit)
        elif p == 'Y':
            qc.sdg(qubit)
            qc.h(qubit)
    qc.measure_all()

    qc_t = transpile(qc, backend=backend, optimization_level=1)
    job = backend.run(qc_t, shots=shots)
    counts = job.result().get_counts()

    expval = 0.0
    total = sum(counts.values())
    for bitstring, count in counts.items():
        parity = sum(int(bitstring[i]) for i, p in enumerate(pauli_label) if p != 'I')
        expval += ((-1) ** (parity % 2)) * count / total
    return expval


def cost_func(params):
    """VQE cost: returns <H> for the given variational parameters."""
    bound = ansatz.assign_parameters(dict(zip(theta, params)))

    energy = 0.0
    for label, coeff in zip(H.paulis.to_labels(), H.coeffs):
        if all(c == 'I' for c in label):
            energy += coeff.real
        else:
            energy += coeff.real * compute_pauli_expectation(bound, label, backend, nshots)
    print("intermediate energy: ", energy)
    return energy

In [14]:
# Optimize
np.random.seed(42)
best_result = None

In [15]:
for trial in range(2):
    x0 = np.random.uniform(0, 2 * np.pi, size=len(theta))
    res = minimize(cost_func, x0, method="cobyla", options={"maxiter": 50, "tol": 1e-2})
    tag = '  <- best so far' if (best_result is None or res.fun < best_result.fun) else ''
    print(f'Trial {trial}: energy = {res.fun:.6f}{tag}')
    if best_result is None or res.fun < best_result.fun:
        best_result = res

intermediate energy:  -0.328
intermediate energy:  -0.712
intermediate energy:  -0.796
intermediate energy:  -0.7280000000000001
intermediate energy:  -0.636
intermediate energy:  -0.74
intermediate energy:  -0.708
intermediate energy:  -0.7200000000000001
intermediate energy:  -0.8160000000000001
intermediate energy:  -0.9199999999999999
intermediate energy:  -0.6479999999999999
intermediate energy:  -0.792
intermediate energy:  -0.8879999999999999
intermediate energy:  -0.88
intermediate energy:  -0.94
intermediate energy:  -0.944
intermediate energy:  -0.9119999999999999
intermediate energy:  -0.9279999999999999
intermediate energy:  -0.9199999999999999
intermediate energy:  -0.972
intermediate energy:  -0.94
intermediate energy:  -0.96
intermediate energy:  -0.9279999999999999
intermediate energy:  -0.94
intermediate energy:  -0.9319999999999999
intermediate energy:  -0.952
intermediate energy:  -0.944
intermediate energy:  -0.9239999999999999
intermediate energy:  -0.8999999999999

In [16]:
print(f"\nOptimized ground state energy : {best_result.fun:.6f}")
print(f"Exact ground state energy     : {eigvals[0]:.6f}")
print(f"Error                         : {abs(best_result.fun - eigvals[0]):.6f}")
print(f"Optimal parameters θ          : {np.round(best_result.x, 4)}")
print(f"Optimization success          : {best_result.success}")


Optimized ground state energy : -0.972000
Exact ground state energy     : -1.000000
Error                         : 0.028000
Optimal parameters θ          : [4.1777 7.2719 4.4855 3.3363 0.8249 0.7943 0.1996 6.5106]
Optimization success          : True
